# Fish Speech Finetune - Debi & Marlene
Google Drive에 fish_speech_data.zip 업로드 후 실행

In [ ]:
# 1. Google Drive 연결
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2. Fish Speech 설치
!git clone https://github.com/fishaudio/fish-speech.git
%cd fish-speech
!pip install -e .

In [ ]:
# 3. 모델 다운로드 (Hugging Face 로그인 필요)
from huggingface_hub import login
login()  # 토큰 입력

In [ ]:
# 4. 모델 다운로드
!python tools/download_models.py

In [ ]:
# 5. 데이터 압축 해제 (Google Drive 경로 수정 필요)
!unzip /content/drive/MyDrive/fish_speech_data.zip -d data/

In [ ]:
# 6. 토큰 추출
!python tools/vqgan/extract_vq.py data \
    --num-workers 1 \
    --batch-size 16 \
    --config-name "modded_dac_vq" \
    --checkpoint-path "checkpoints/openaudio-s1-mini/codec.pth"

In [ ]:
# 7. 데이터셋 변환
!python tools/llama/build_dataset.py \
    --input "data" \
    --output "data/protos" \
    --text-extension .lab \
    --num-workers 16

In [ ]:
# 8. 파인튜닝 (LoRA)
!python fish_speech/train.py --config-name text2semantic_finetune \
    trainer.max_steps=3000

In [ ]:
# 9. 학습된 모델을 Google Drive에 저장
!cp -r results /content/drive/MyDrive/fish_speech_results